In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pathogen_atlas_v3").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)


# Phase 2 · PubMed searches and abstracts

This notebook shows the selected organism groups, taxonomy inputs, cache/release locations, pending searches, and a lower-bound ETA before any NCBI request. It retains the full PMID search results for counts, then selects up to 200 PMIDs per species for abstract fetch and scoring: the 30 lowest numeric PMIDs followed by a reproducible random sample (seed 42). The `batch_size` of 200 controls PMIDs per EFetch request, not the per-species sample. The named `run_phase2` function expands over-cap PubMed searches with resumable date/UID segments, fetches selected PMIDs once, appends raw cache records, and checkpoints normalized Parquet releases. Reruns reuse cached queries, overflow segments, and articles.

In [2]:
from pathlib import Path
import inspect
import pandas as pd
from IPython.display import display
from pathogen_atlas_v3.abstracts.pubmed import resolve_ncbi_email, _ncbi_key_from_keychain, acquire_pubmed
from pathogen_atlas_v3.common.config import selected_organism_groups
from pathogen_atlas_v3.common.io import load_yaml
from pathogen_atlas_v3.pipelines.phase2_pubmed import phase2_preflight, run_phase2

CONFIG_PATH = "configs/default.yaml"
CONFIG = load_yaml(CONFIG_PATH)
GROUPS = selected_organism_groups(CONFIG)
TAXONOMY_ROOT = Path(CONFIG["paths"]["taxonomy_release"])
SPECIES = pd.read_parquet(TAXONOMY_ROOT / "species.parquet")
ALIASES = pd.read_parquet(TAXONOMY_ROOT / "aliases.parquet")
print("Selected groups:", GROUPS)
print("NCBI contact available:", bool(resolve_ncbi_email()))
print("NCBI API key available in environment or Keychain:", bool(_ncbi_key_from_keychain()))
print("Workers:", CONFIG["pubmed"]["max_workers"], "| request delay:", CONFIG["pubmed"]["request_delay_seconds"], "| retmax:", CONFIG["pubmed"]["retmax"])
print("Per-species sampling:", CONFIG["pubmed"].get("sampling", {}), "| EFetch batch size:", CONFIG["pubmed"]["batch_size"])
display(SPECIES.loc[SPECIES["organism_group"].isin(GROUPS)].groupby("organism_group").size().rename("taxonomy species").to_frame())
print("\nImplementation called by run_phase2:\n")
print(inspect.getsource(run_phase2))

Selected groups: ['fungi']
NCBI contact available: True
NCBI API key available in environment or Keychain: True
Workers: 10 | request delay: None | retmax: 10000
Per-species sampling: {} | EFetch batch size: 200


,taxonomy species
organism_group,
fungi,101843



Implementation called by run_phase2:

def run_phase2(config_path: str = "configs/default.yaml", *, max_calls: int | None = None, max_fetch_batches: int | None = None, log: bool = True):
    config = load_yaml(config_path)
    species = pd.read_parquet(config["paths"]["taxonomy_release"] + "/species.parquet")
    aliases = pd.read_parquet(config["paths"]["taxonomy_release"] + "/aliases.parquet")
    settings, paths = config["pubmed"], config["paths"]
    sampling = settings.get("sampling", {})
    groups = selected_organism_groups(config)
    root = Path(paths.get("pubmed_domain_root", Path(paths["pubmed_release"]).parent / "domains"))
    request_limit = max_calls if max_calls is not None else settings.get("max_calls")
    fetch_limit = max_fetch_batches if max_fetch_batches is not None else settings.get("max_fetch_batches")
    progress_every = int(config.get("logging", {}).get("progress_every", 100))
    results = []
    remaining_calls = None if request_limit is None else int(reque

## Preflight · no network calls

Check the pending query count and ETA before enabling the run. The ETA estimates only ESearch pacing; retries, PubMed result volumes, and abstract retrieval add time. Query and fetch progress logs print current completion and ETA at the interval configured by `logging.progress_every`.

In [3]:
PREFLIGHT = phase2_preflight(CONFIG_PATH)
display(PREFLIGHT)
CACHE_AND_RELEASE = pd.DataFrame([{"group": row["organism_group"], "species": row["species"], "cached queries": row["cached_queries"], "pending queries": row["pending_queries"], "cached over-cap queries": row["incomplete_cached_queries"], "overflow segments": row["cached_overflow_segments"], "cache": row["cache_path"], "release": str(Path(CONFIG["paths"]["pubmed_domain_root"]) / row["organism_group"] / "release")} for row in PREFLIGHT["per_group"]])
display(CACHE_AND_RELEASE)

{'status': 'ready',
 'selected_groups': ['fungi'],
 'per_group': [{'organism_group': 'fungi',
   'species': 101843,
   'cached_queries': 32143,
   'pending_queries': 69664,
   'species_without_searchable_names': 36,
   'incomplete_cached_queries': 0,
   'cached_overflow_segments': 0,
   'cache_path': 'data/02_pubmed/domains/fungi/cache/pubmed_search_results.jsonl'}],
 'pending_queries': 69664,
 'excluded_pubmed_aliases': {'ictv_isolate_designations': 8034,
  'configured_ambiguous_aliases': 0,
  'fungal_taxonomy_ambiguities': 383},
 'request_delay_seconds': 0.125,
 'estimated_search_floor_minutes': 145.1,
 'retmax_per_query': 10000,
 'sampling': {'max_pmids_per_species': 200,
  'oldest_pmids_per_species': 30,
  'random_seed': 42},
 'estimated_search_note': 'Lower-bound new ESearch time; excludes retries, overflow expansion, throttling, and abstract retrieval.'}

,group,species,cached queries,pending queries,cached over-cap queries,overflow segments,cache,release
0,fungi,101843,32143,69664,0,0,data/02_pubmed/domains/fungi/cache/pubmed_sear...,data/02_pubmed/domains/fungi/release


## Run Phase 2

Set `MAX_QUERY_CALLS` and `MAX_FETCH_BATCHES` to small values for a bounded run. Set them to `None` for the full selected-domain acquisition. Interrupting the notebook is safe: completed searches, overflow segments, and abstracts are cached; the same call resumes.

In [4]:
RUN_PHASE = True
MAX_QUERY_CALLS = None  # full selected-domain search; cache makes reruns resumable
MAX_FETCH_BATCHES = None  # fetch every missing abstract batch
LOG_PROGRESS = True

if RUN_PHASE:
    if not resolve_ncbi_email():
        raise RuntimeError("Set NCBI_EMAIL or configure the local Git author email before NCBI requests.")
    MATCHES, ABSTRACTS = run_phase2(CONFIG_PATH, max_calls=MAX_QUERY_CALLS, max_fetch_batches=MAX_FETCH_BATCHES, log=LOG_PROGRESS)
    sampled_rows = int(MATCHES["selected_for_sampling"].sum()) if "selected_for_sampling" in MATCHES else len(MATCHES)
    fetched_rows = int(MATCHES["selected_for_fetch"].sum()) if "selected_for_fetch" in MATCHES else len(MATCHES)
    print(f"Phase 2 retained {len(MATCHES):,} full search associations; {sampled_rows:,} sampled for fetch; {fetched_rows:,} have fetched abstracts; {len(ABSTRACTS):,} sampled unique abstracts stored.")
else:
    print("Preflight only. Set RUN_PHASE=True to run the selected organism groups.")

Phase 2 · fungi: 101,843 taxa; aliases 129,228; cache data/02_pubmed/domains/fungi/cache; release data/02_pubmed/domains/fungi/release.
PubMed search: 101,843 species rows; excluded aliases: 0 ICTV designations, 0 configured terms, 383 ambiguous fungal mappings; 36 taxa without searchable names; 32,143 cached queries; 69,664 pending.
PubMed search: 32,243/101,807 (31.7%); 2,027 species with filtered PMID hits; 3,283 unique filtered PMIDs collected; ETA 4 hours 14 minutes.
PubMed search: 32,343/101,807 (31.8%); 2,030 species with filtered PMID hits; 3,288 unique filtered PMIDs collected; ETA 4 hours 12 minutes.
PubMed search: 32,443/101,807 (31.9%); 2,041 species with filtered PMID hits; 3,303 unique filtered PMIDs collected; ETA 4 hours 44 minutes.
PubMed search: 32,543/101,807 (32.0%); 2,049 species with filtered PMID hits; 3,310 unique filtered PMIDs collected; ETA 4 hours 10 minutes.
PubMed search: 32,643/101,807 (32.1%); 2,054 species with filtered PMID hits; 3,314 unique filtered 

/Users/f.costa/Documents/Projects/Repositories/ACTIVE/Pathogen Atlas/Pathogen Atlas/Pathogen Atlas Data Acquisition/version3/pathogen_atlas_v3/pipelines/phase2_pubmed.py:163: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  abstract_matches = preserved_matches.loc[preserved_matches["selected_for_sampling"].fillna(False).astype(bool)]


Phase 2 retained 44,354 full search associations; 23,252 sampled for fetch; 22,427 have fetched abstracts; 19,115 sampled unique abstracts stored.
